# NB05 · Semana 10 — El DAG en Airflow (Entrega 3)
**Ingeniería de Datos — LCD · UCA Rosario · 2026** · grupo06 · Clima de Sídney (Open-Meteo)

`dags/pipeline_grupo06.py` encadena cuatro tareas atómicas:

```
preparar_entorno ─► extraer_bronze ─► dbt_build ─► publicar_gold
```
| Tarea | Qué hace | Por qué separada |
|---|---|---|
| `preparar_entorno` | Crea (una vez) un venv propio con deltalake/dbt | El Airflow es compartido y no trae esas librerías; si falla es infraestructura, no datos |
| `extraer_bronze` | Open-Meteo → bronze para el intervalo de la corrida | Depende de una API externa: es la que más reintentos necesita |
| `dbt_build` | silver + gold + tests | Determinística: si un test falla, no se publica |
| `publicar_gold` | DuckDB → PostgreSQL | Toca el warehouse que ve Metabase; reintentable sola |

In [ ]:
import os, sys
from pathlib import Path
# Raíz del proyecto = carpeta padre de notebooks/ (en el servidor: ~/work/dags)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
import entorno
entorno.info()
print((RAIZ / "dags" / "pipeline_grupo06.py").read_text())

## 1. Validar el DAG antes de desplegar
Sintaxis (`ast`) + estructura: tareas, dependencias y que la extracción use el intervalo y no el reloj.

In [ ]:
import ast
codigo = (RAIZ / "dags" / "pipeline_grupo06.py").read_text()
arbol = ast.parse(codigo); compile(codigo, "pipeline_grupo06.py", "exec")
print("✔ sintaxis OK")
ids = [kw.value.value for n in ast.walk(arbol) if isinstance(n, ast.Call)
       for kw in n.keywords if kw.arg == "task_id"]
print("tareas:", ids)
assert ids == ["preparar_entorno", "extraer_bronze", "dbt_build", "publicar_gold"]
assert "data_interval_start" in codigo and "data_interval_end" in codigo
print("✔ la extracción recibe el intervalo del contexto")

## 2. Cómo se ve una corrida
Para la corrida con fecha lógica `2026-03-10 06:00` (ART), Airflow arma el comando:
```
python -m scripts.ingesta --inicio 2026-03-10 --fin 2026-03-11
```
y la ingesta procesa el **10/03 de Sídney**, aunque la ejecutemos hoy. Probamos ese mismo comando acá:

In [ ]:
import subprocess
r = subprocess.run("python -m scripts.ingesta --inicio 2026-03-10 --fin 2026-03-11", shell=True, cwd=RAIZ, capture_output=True, text=True)
print(r.stdout, r.stderr[-1500:])

## 3. Backfill (cómo dispararlo)
**Un día o pocos días** — en la UI de Airflow: *Trigger DAG w/ config* (▶ → *Trigger DAG w/ config*) → elegir **Logical date** en el pasado → Trigger.
Cada corrida reprocesa exactamente su intervalo (MERGE: no duplica).

**Un rango largo en una sola corrida** — mismo botón, con config:
```json
{"inicio": "2026-03-01", "fin": "2026-04-01"}
```
La ingesta pide el rango completo a la API (un pedido por año), en vez de 31 corridas.

**¿Y backfillear dos años?** Con corridas diarias serían ~730 corridas × 4 tareas (≈3000 tareas en un Airflow compartido) y 730 pedidos a la API.
Lo correcto es **una corrida con rango** (o pocas, por año): Open-Meteo acepta rangos largos y nuestro `dbt build` es completo, no incremental, así que basta con correrlo una vez al final.

`max_active_runs=1`: dos corridas a la vez escribirían la misma tabla Delta y el mismo archivo DuckDB.